In [1]:
import pandas as pd
from datetime import datetime

In [2]:
# 1. Load the CSV file
df = pd.read_csv("sap_user_logon_export.csv")

# ORIGINAL DF as read from .csv file
df 

,UserID,UserType,LastLogon,Locked,Role,Manager
0,RC1001,Dialog,2025-02-10,No,Z_AP_CLERK,Smith
1,RC1002,Dialog,2024-10-01,No,Z_AR_CLERK,Smith
2,RC1003,Dialog,2024-06-15,No,SAP_ALL,Jones
3,RC1004,Dialog,2025-01-05,Yes,Z_DISPLAY_FIN,Jones
4,BATCH01,System,2025-02-18,No,Z_BATCH_JOB,NaN
5,INTF01,Communication,2025-02-20,No,Z_INTERFACE,NaN
6,RC1005,Dialog,2024-08-20,No,Z_MM_USER,Lee
7,RC1006,Dialog,2023-12-01,No,Z_HR_PAYROLL,Lee
8,RC1007,Dialog,2025-02-01,No,Z_DISPLAY_FIN,Smith
9,RC1008,Dialog,2024-11-15,No,Z_ADMIN_ROLE,Jones


In [3]:
# ORIGINAL DF info + stats
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   UserID     10 non-null     object
 1   UserType   10 non-null     object
 2   LastLogon  10 non-null     object
 3   Locked     10 non-null     object
 4   Role       10 non-null     object
 5   Manager    8 non-null      object
dtypes: object(6)
memory usage: 612.0+ bytes


In [4]:
# 2. Convert LastLogon column to datetime format
df["LastLogon"] = pd.to_datetime(df["LastLogon"])

# Check Dtype of LastLogon column after conversion - Dtype of LastLogon should now be datetime format!
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   UserID     10 non-null     object        
 1   UserType   10 non-null     object        
 2   LastLogon  10 non-null     datetime64[ns]
 3   Locked     10 non-null     object        
 4   Role       10 non-null     object        
 5   Manager    8 non-null      object        
dtypes: datetime64[ns](1), object(5)
memory usage: 612.0+ bytes


In [5]:
# 3. Calculate inactivity days
today = datetime.today() # store todays date in variable "today"
print(today)

# Add column "DaysInactive" with number of days since user last active on system
# .dt.days extracts actual number of DAYS from a complete date and timestamp signature
# notice parentheses containing subtraction operation between today's date and LastLogon date
# Order of Operation MUST be observed!
# ALSO notice LastLogon date is subtracted from todays date - this appears to be more accurate 
#in preventing an "off-by-one" issue where 1 additional day is added to days inactive!

df["DaysInactive"] = (today - df["LastLogon"]).dt.days
df["DaysInactive"]

2026-02-28 06:12:48.004681


0    383
1    515
2    623
3    419
4    375
5    373
6    557
7    820
8    392
9    470
Name: DaysInactive, dtype: int64

In [6]:
# DF now has DaysInactive column
df.head()

,UserID,UserType,LastLogon,Locked,Role,Manager,DaysInactive
0,RC1001,Dialog,2025-02-10,No,Z_AP_CLERK,Smith,383
1,RC1002,Dialog,2024-10-01,No,Z_AR_CLERK,Smith,515
2,RC1003,Dialog,2024-06-15,No,SAP_ALL,Jones,623
3,RC1004,Dialog,2025-01-05,Yes,Z_DISPLAY_FIN,Jones,419
4,BATCH01,System,2025-02-18,No,Z_BATCH_JOB,NaN,375


In [7]:
# DO NOT DELETE THIS!
# # This operation returns a Timedelta object to variable x
# # https://pandas.pydata.org/docs/reference/api/pandas.Timedelta.html
# x = df["LastLogon"][0] - today

# # Extract NUMBER of days from Timedelta object via .days() method
# # https://pandas.pydata.org/docs/reference/api/pandas.Timedelta.days.html#pandas.Timedelta.days
# x.days

# # x2 = today - df["LastLogon"][0]
# # x2.days

In [8]:
# 4. Remove System and Communication users
df = df[~df["UserType"].isin(["System", "Communication"])]
# df
# df = df.mask(df["UserType"].str.contains("System|Communication", case=False, na=False))

In [9]:
# NOTE ALL SYSTEM and COMMUNICATION USERS HAVE BEEN REMOVED!
df

,UserID,UserType,LastLogon,Locked,Role,Manager,DaysInactive
0,RC1001,Dialog,2025-02-10,No,Z_AP_CLERK,Smith,383
1,RC1002,Dialog,2024-10-01,No,Z_AR_CLERK,Smith,515
2,RC1003,Dialog,2024-06-15,No,SAP_ALL,Jones,623
3,RC1004,Dialog,2025-01-05,Yes,Z_DISPLAY_FIN,Jones,419
6,RC1005,Dialog,2024-08-20,No,Z_MM_USER,Lee,557
7,RC1006,Dialog,2023-12-01,No,Z_HR_PAYROLL,Lee,820
8,RC1007,Dialog,2025-02-01,No,Z_DISPLAY_FIN,Smith,392
9,RC1008,Dialog,2024-11-15,No,Z_ADMIN_ROLE,Jones,470


In [10]:
# 5. Remove already locked users
# df = df[df["Locked"] == "No"]

# A "MUCH CLEANER ALTERNATIVE" to above syntax
df = df.query("Locked == 'No'") # Note how the expression is wrapped in the quotaion marks!

In [11]:
df # Notice locked users have been removed!

,UserID,UserType,LastLogon,Locked,Role,Manager,DaysInactive
0,RC1001,Dialog,2025-02-10,No,Z_AP_CLERK,Smith,383
1,RC1002,Dialog,2024-10-01,No,Z_AR_CLERK,Smith,515
2,RC1003,Dialog,2024-06-15,No,SAP_ALL,Jones,623
6,RC1005,Dialog,2024-08-20,No,Z_MM_USER,Lee,557
7,RC1006,Dialog,2023-12-01,No,Z_HR_PAYROLL,Lee,820
8,RC1007,Dialog,2025-02-01,No,Z_DISPLAY_FIN,Smith,392
9,RC1008,Dialog,2024-11-15,No,Z_ADMIN_ROLE,Jones,470


In [12]:
# SEE THIS LINK for info regarding this syntax:
# https://gemini.google.com/share/652021612b62
# https://gemini.google.com/share/1d987cb02d3a (LATEST + more detailed explanation re: ValueError: Columns must be same length as key)

# 6. Flag users inactive more than 90 days- returns boolean values (boolean masking/ filtering)
# df["InactiveFlag"] = df["DaysInactive"] > 90 # works but raises "SettingWithCopyError" / "SettingWithCopyWarning:"

# Works: Raises warning FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated.
# df["DaysInactive"].mask(df["DaysInactive"] > 90, df["InactiveFlag"]) 

# Using df.loc[] to assign values to new column in DF ( WORKING EXAMPLES w/ no errors/ warnings)
# SYNTAX: df.loc[row_indexer, newColumnName] = someValue (new column created if does not already exist in DF)

# Ex 1 - Using slice as row indexer
df.loc[:, "InactiveFlag"] = df["DaysInactive"] > 90 

# Ex 2 - Using an explicit label as row indexer
# In this case the label df["DaysInactive"] > 90 tells df.loc to locate all rows in DaysInactive column greater than 90
# df.loc[df["DaysInactive"] > 90, "InactiveFlag"] = True

# Boolean value now written into new colum "InactiveFlag"
df

# Refer to below path for syntax examples for .mask
# http://localhost:8889/notebooks/OneDrive/Desktop/pandas_example_notebooks/replacing_values_with_boolean_masking(dirty_cafe_sales_example).ipynb

C:\Users\Cessn\AppData\Local\Temp\ipykernel_13104\2831096767.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:, "InactiveFlag"] = df["DaysInactive"] > 90


,UserID,UserType,LastLogon,Locked,Role,Manager,DaysInactive,InactiveFlag
0,RC1001,Dialog,2025-02-10,No,Z_AP_CLERK,Smith,383,True
1,RC1002,Dialog,2024-10-01,No,Z_AR_CLERK,Smith,515,True
2,RC1003,Dialog,2024-06-15,No,SAP_ALL,Jones,623,True
6,RC1005,Dialog,2024-08-20,No,Z_MM_USER,Lee,557,True
7,RC1006,Dialog,2023-12-01,No,Z_HR_PAYROLL,Lee,820,True
8,RC1007,Dialog,2025-02-01,No,Z_DISPLAY_FIN,Smith,392,True
9,RC1008,Dialog,2024-11-15,No,Z_ADMIN_ROLE,Jones,470,True


In [24]:
# # 7. Flag high-risk roles
# df["HighRiskRole"] = df["Role"].str.contains("SAP_ALL|ADMIN", case=False, na=False)

# TRY THIS ONE via alternate means:
# df["HighRiskRole"] = df.loc[df["Role"].str.contains("SAP_ALL|ADMIN", case=False, na=False)] # NO GOOD
df.loc[:, "HighRiskRole"] = df["Role"].str.contains("SAP_ALL|ADMIN", case=False, na=False) # THIS WORKS!
df

# df["Role"].mask(df["Role"].str.contains("sap_all|admin"), case=False, na=False), df["HighRiskRole"]

C:\Users\Cessn\AppData\Local\Temp\ipykernel_21068\14229372.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:, "HighRiskRole"] = df["Role"].str.contains("SAP_ALL|ADMIN", case=False, na=False) # THIS WORKS!


,UserID,UserType,LastLogon,Locked,Role,Manager,DaysInactive,InactiveFlag,HighRiskRole
0,RC1001,Dialog,2025-02-10,No,Z_AP_CLERK,Smith,383,True,False
1,RC1002,Dialog,2024-10-01,No,Z_AR_CLERK,Smith,515,True,False
2,RC1003,Dialog,2024-06-15,No,SAP_ALL,Jones,623,True,True
6,RC1005,Dialog,2024-08-20,No,Z_MM_USER,Lee,557,True,False
7,RC1006,Dialog,2023-12-01,No,Z_HR_PAYROLL,Lee,820,True,False
8,RC1007,Dialog,2025-02-01,No,Z_DISPLAY_FIN,Smith,392,True,False
9,RC1008,Dialog,2024-11-15,No,Z_ADMIN_ROLE,Jones,470,True,True


In [26]:
# 8. Keep only users needing review
review_df = df[(df["InactiveFlag"] == True) | (df["HighRiskRole"] == True)]

In [27]:
review_df

,UserID,UserType,LastLogon,Locked,Role,Manager,DaysInactive,InactiveFlag,HighRiskRole
0,RC1001,Dialog,2025-02-10,No,Z_AP_CLERK,Smith,382,True,False
1,RC1002,Dialog,2024-10-01,No,Z_AR_CLERK,Smith,514,True,False
2,RC1003,Dialog,2024-06-15,No,SAP_ALL,Jones,622,True,True
6,RC1005,Dialog,2024-08-20,No,Z_MM_USER,Lee,556,True,False
7,RC1006,Dialog,2023-12-01,No,Z_HR_PAYROLL,Lee,819,True,False
8,RC1007,Dialog,2025-02-01,No,Z_DISPLAY_FIN,Smith,391,True,False
9,RC1008,Dialog,2024-11-15,No,Z_ADMIN_ROLE,Jones,469,True,True


In [ ]:
# 9. Export the report
review_df.to_excel("lock_recommendation_report.xlsx", index=False)
print("Lock Recommendation Report Generated Successfully.")